## 导入模型

In [1]:
import os
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from tools.tools import build_model_theory_torch

model_name = os.environ.get("PARAMEST_MODEL_NAME", "tls").strip().lower()
retrain = False
load_training_data = False

waiting_time_real_torch, laplace_transform_torch = build_model_theory_torch(model_name)

%run ./2-train.ipynb


/space-nobackup/ge48zus/venvs/paramestnn/lib/python3.12/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Using device: cpu
model_name=tls
Loaded checkpoint only, skipped dataset load: /home/t30/rab/ge48zus/paramest-nn/ParamEst-NN-main/notebooks/checkpoints/pole_residue_block_head_tls.pt
batch_size=1024, num_epochs=4, train_length_choices=[10, 15, 20, 30, 40, 50]
alpha_init_range=(0.1, 1.5), beta_init_range=(-5.0, 5.0)
target_mean = tensor([1.4645, 2.6470])
target_std  = tensor([0.8648, 1.3777])
beta_init range = (-5.0, 5.0)
feature_dim = 7
retrain = False
checkpoint = /home/t30/rab/ge48zus/paramest-nn/ParamEst-NN-main/notebooks/checkpoints/pole_residue_block_head_tls.pt
Loaded model checkpoint from /home/t30/rab/ge48zus/paramest-nn/ParamEst-NN-main/notebooks/checkpoints/pole_residue_block_head_tls.pt
Best validation standardized MSE: 0.574648
Best validation RMSE [delta, omega]: [0.7425797  0.88429594]
Learned alpha: [0.53396785 0.09950528 1.1093745 ]
Learned beta : [-4.3752995   0.33912888  5.0392103 ]
Effective standardized weight rows:
tensor([[-0.9662, -1.0743,  2.0480, -0.8274, -3.07

/tmp/ipykernel_2112375/519262741.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt_for_load = torch.load(model_ckpt_path, map_location="cpu") if model_ckpt_path.exis

## Dense Scan


### setup

In [ ]:
L_grid = train_length_choices

delta_range = (0.10, 3.00)
omega_range = (0.50, 5.00)
n_delta = int(os.environ.get("PARAMEST_N_DELTA", "11"))
n_omega = int(os.environ.get("PARAMEST_N_OMEGA", "11"))
tau_stride = max(1, int(os.environ.get("PARAMEST_TAU_STRIDE", "1")))
progress_every = max(1, int(os.environ.get("PARAMEST_PROGRESS_EVERY", "12")))

delta_values = np.linspace(*delta_range, n_delta)
omega_values = np.linspace(*omega_range, n_omega)
theta_grid = [[float(delta), float(omega)] for delta in delta_values for omega in omega_values]

if tau_stride > 1:
    tau_grid = tau_grid[::tau_stride].contiguous()

H_test = as_real_tensor(
    [
        [0.02, 0.00],
        [-0.02, 0.00],
        [0.00, 0.05],
        [0.00, -0.05],
    ],
    device=device,
)

print("L_grid =", L_grid)
print("delta range =", (float(delta_values.min()), float(delta_values.max()), len(delta_values)))
print("omega range =", (float(omega_values.min()), float(omega_values.max()), len(omega_values)))
print("n_theta =", len(theta_grid))
print("tau_grid points =", int(tau_grid.numel()), "(tau_stride =", tau_stride, ")")
print("H_test =")
print(tensor_to_numpy(H_test))

scan_mode = "serial"
print_scan_table_enabled = os.environ.get("PARAMEST_PRINT_SCAN_TABLE", "0") == "1"
print("scan_mode =", scan_mode)
print("print_scan_table =", print_scan_table_enabled)
print("progress_every =", progress_every)

figure_dir = PROJECT_ROOT / "notebooks" / "figures" / model_name
figure_dir.mkdir(parents=True, exist_ok=True)
print("figure_dir =", figure_dir)

scan_cache_dir = PROJECT_ROOT / "notebooks" / "scan_cache" / model_name
scan_cache_dir.mkdir(parents=True, exist_ok=True)
scan_df_csv_path = scan_cache_dir / f"{model_name}_scan_df.csv"
print("scan_df_csv_path =", scan_df_csv_path)


### Scan Helpers


In [ ]:
# One independent (L, theta) task executed serially

SCORE_FD_STEP = float(os.environ.get("PARAMEST_SCORE_FD_STEP", "1e-3"))
JAC_FD_STEP = float(os.environ.get("PARAMEST_JAC_FD_STEP", "1e-3"))

def score_wrt_theta_on_grid(theta, tau_grid, w_func, eps_w=EPS_W, renorm=True):
    theta = as_real_tensor(theta, device=tau_grid.device)
    d = int(theta.numel())
    n = int(tau_grid.numel())
    scores = torch.zeros((d, n), device=theta.device, dtype=theta.dtype)

    for k in range(d):
        h = torch.zeros_like(theta)
        h[k] = SCORE_FD_STEP
        w_plus = normalized_w_on_grid(theta + h, tau_grid, w_func, eps=eps_w, renorm=renorm)
        w_minus = normalized_w_on_grid(theta - h, tau_grid, w_func, eps=eps_w, renorm=renorm)
        scores[k] = (torch.log(torch.clamp(w_plus, min=eps_w)) - torch.log(torch.clamp(w_minus, min=eps_w))) / (2.0 * SCORE_FD_STEP)

    return scores

def jac_eta_wrt_theta(theta, model, N_obs, tau_grid, w_func, laplace_w=None, eps_w=EPS_W, renorm=True):
    theta = as_real_tensor(theta)
    d = int(theta.numel())
    J = torch.zeros((d, d), device=theta.device, dtype=theta.dtype)

    for k in range(d):
        h = torch.zeros_like(theta)
        h[k] = JAC_FD_STEP
        eta_plus = estimator_mean_linear_laplace(
            model=model,
            theta=theta + h,
            N_obs=N_obs,
            tau_grid=tau_grid,
            w_func=w_func,
            laplace_w=laplace_w,
            eps_w=eps_w,
            renorm=renorm,
        )
        eta_minus = estimator_mean_linear_laplace(
            model=model,
            theta=theta - h,
            N_obs=N_obs,
            tau_grid=tau_grid,
            w_func=w_func,
            laplace_w=laplace_w,
            eps_w=eps_w,
            renorm=renorm,
        )
        J[:, k] = (eta_plus - eta_minus) / (2.0 * JAC_FD_STEP)

    return J

def _compute_scan_row(model, theta, L, tau_grid, w_func, laplace_w, H_test, device=None):
    device = device or next(model.parameters()).device
    theta_t = as_real_tensor(theta, device=device)

    exact_report = linear_laplace_estimator_report(
        model=model,
        theta=theta_t,
        N_obs=L,
        tau_grid=tau_grid,
        w_func=w_func,
        laplace_w=laplace_w,
        jac_eta_func=jac_eta_wrt_theta,
        score_func=score_wrt_theta_on_grid,
    )

    eta_func = lambda th: estimator_mean_linear_laplace(
        model=model,
        theta=th,
        N_obs=L,
        tau_grid=tau_grid,
        w_func=w_func,
        laplace_w=laplace_w,
    )

    bar_report = generalized_barankin_bound(
        theta=theta_t,
        H=H_test,
        N_obs=L,
        tau_grid=tau_grid,
        w_func=w_func,
        eta_func=eta_func,
    )

    exact_mse_diag = torch.diag(exact_report["MSE_exact"])
    bcrb_mse_diag = torch.diag(exact_report["MSE_bCRB"])
    bar_mse_diag = torch.diag(bar_report["MSE_barankin"])

    row = {
        "L": int(L),
        "delta": float(theta_t[0].detach().cpu()),
        "omega": float(theta_t[1].detach().cpu()),
        "exact_mse_delta": float(exact_mse_diag[0].detach().cpu()),
        "bcrb_mse_delta": float(bcrb_mse_diag[0].detach().cpu()),
        "barankin_mse_delta": float(bar_mse_diag[0].detach().cpu()),
        "exact_mse_omega": float(exact_mse_diag[1].detach().cpu()),
        "bcrb_mse_omega": float(bcrb_mse_diag[1].detach().cpu()),
        "barankin_mse_omega": float(bar_mse_diag[1].detach().cpu()),
    }
    row["exact_over_bcrb_delta"] = row["exact_mse_delta"] / max(row["bcrb_mse_delta"], 1e-12)
    row["exact_over_barankin_delta"] = row["exact_mse_delta"] / max(row["barankin_mse_delta"], 1e-12)
    row["exact_over_bcrb_omega"] = row["exact_mse_omega"] / max(row["bcrb_mse_omega"], 1e-12)
    row["exact_over_barankin_omega"] = row["exact_mse_omega"] / max(row["barankin_mse_omega"], 1e-12)
    return row


def scan_over_lengths_and_points(
    model,
    theta_points,
    L_list,
    tau_grid,
    w_func,
    laplace_w,
    H_test,
    device=None,
    print_table=False,
):
    device = device or next(model.parameters()).device
    all_rows = []

    for L in L_list:
        total_points = len(theta_points)
        print(f"Starting L = {int(L)} with {total_points} theta points ...")
        start_t = time.perf_counter()
        rows = []
        for idx, theta in enumerate(theta_points, start=1):
            rows.append(
                _compute_scan_row(
                    model=model,
                    theta=theta,
                    L=L,
                    tau_grid=tau_grid,
                    w_func=w_func,
                    laplace_w=laplace_w,
                    H_test=H_test,
                    device=device,
                )
            )
            if idx % progress_every == 0 or idx == total_points:
                elapsed = time.perf_counter() - start_t
                print(f"  progress L={int(L)}: {idx}/{total_points}, elapsed={elapsed:.1f}s")
        rows.sort(key=lambda row: (row["delta"], row["omega"]))
        all_rows.extend(rows)
        elapsed = time.perf_counter() - start_t
        print(f"Finished L = {int(L)} in {elapsed:.2f}s")
        if print_table:
            print_scan_table(rows)
        print()

    all_rows.sort(key=lambda row: (row["L"], row["delta"], row["omega"]))
    return all_rows


def print_scan_table(rows):
    header = (
        f"{'L':>3} {'delta':>7} {'omega':>7} "
        f"{'r_d/b':>10} {'r_d/bar':>10} "
        f"{'r_o/b':>10} {'r_o/bar':>10}"
    )
    print(header)
    print("-" * len(header))
    for row in rows:
        print(
            f"{row['L']:3d} "
            f"{row['delta']:7.3f} {row['omega']:7.3f} "
            f"{row['exact_over_bcrb_delta']:10.4f} {row['exact_over_barankin_delta']:10.4f} "
            f"{row['exact_over_bcrb_omega']:10.4f} {row['exact_over_barankin_omega']:10.4f}"
        )


def scan_rows_to_frame(scan_rows):
    df = pd.DataFrame(scan_rows)
    if df.empty:
        return df
    return df.sort_values(["L", "delta", "omega"]).reset_index(drop=True)


def make_ratio_grid(df_L, ratio_key):
    pivot = df_L.pivot(index="omega", columns="delta", values=ratio_key).sort_index().sort_index(axis=1)
    delta_vals = pivot.columns.to_numpy(dtype=float)
    omega_vals = pivot.index.to_numpy(dtype=float)
    ratio_grid = pivot.to_numpy(dtype=float)
    return delta_vals, omega_vals, ratio_grid


def plot_ratio_surfaces(scan_df, ratio_keys, z_titles, figure_title, cmap="viridis"):
    if scan_df.empty:
        raise ValueError("scan_df is empty.")

    L_values = sorted(scan_df["L"].unique())
    nrows = len(ratio_keys)
    ncols = len(L_values)
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(4.2 * ncols, 3.8 * nrows),
        constrained_layout=True,
        squeeze=False,
    )

    for row_idx, (ratio_key, z_title) in enumerate(zip(ratio_keys, z_titles)):
        for col_idx, L in enumerate(L_values):
            ax = axes[row_idx][col_idx]
            df_L = scan_df.loc[scan_df["L"] == L, ["delta", "omega", ratio_key]].copy()
            delta_vals, omega_vals, ratio_grid = make_ratio_grid(df_L, ratio_key)
            mesh = ax.pcolormesh(delta_vals, omega_vals, ratio_grid, shading="auto", cmap=cmap)
            ax.set_title(f"L = {L} | {z_title}")
            ax.set_xlabel("delta")
            ax.set_ylabel("omega")
            cbar = fig.colorbar(mesh, ax=ax, shrink=0.85, pad=0.02)
            cbar.set_label("ratio")

    fig.suptitle(figure_title, fontsize=16)
    return fig


## Per-L Tables & Figures


### tables

In [ ]:
scan_rows = scan_over_lengths_and_points(
    model=model,
    theta_points=theta_grid,
    L_list=L_grid,
    tau_grid=tau_grid,
    w_func=w_func,
    laplace_w=laplace_w,
    H_test=H_test,
    device=device,
    print_table=print_scan_table_enabled,
)

scan_df = scan_rows_to_frame(scan_rows)
scan_df.to_csv(scan_df_csv_path, index=False)
print("Saved scan_df to", scan_df_csv_path)
print("n_rows =", len(scan_rows))
display(scan_df.head())


### Learned / bCRB 3D Surfaces


In [ ]:
fig_bcrb = plot_ratio_surfaces(
    scan_df=scan_df,
    ratio_keys=["exact_over_bcrb_delta", "exact_over_bcrb_omega"],
    z_titles=["delta: learned / bCRB", "omega: learned / bCRB"],
    figure_title=f"{model_name} | learned / bCRB ratio surfaces",
    cmap="viridis",
)
fig_bcrb.savefig(figure_dir / f"{model_name}_learned_over_bcrb_surfaces.png", dpi=220, bbox_inches="tight")
plt.show()


### Learned / Barankin 3D Surfaces


In [ ]:
fig_barankin = plot_ratio_surfaces(
    scan_df=scan_df,
    ratio_keys=["exact_over_barankin_delta", "exact_over_barankin_omega"],
    z_titles=["delta: learned / Barankin", "omega: learned / Barankin"],
    figure_title=f"{model_name} | learned / Barankin ratio surfaces",
    cmap="magma",
)
fig_barankin.savefig(figure_dir / f"{model_name}_learned_over_barankin_surfaces.png", dpi=220, bbox_inches="tight")
plt.show()
